# SAM-Med3D Hybrid for OAI-ZIB Knee MRI

This notebook implements a **hybrid SAM-Med3D pipeline**:

- **Bone model @1.5mm**: use the baseline/all-class checkpoint for bone classes.
- **Cartilage model @0.7mm**: fine-tune only the cartilage classes for **100 epochs**.
- **Hybrid prediction**: combine bone predictions from the 1.5mm model with cartilage predictions from the 0.7mm model.

Class names are written explicitly everywhere:

| Label | Class name | Group |
|---:|---|---|
| 1 | femoral_bone | bone |
| 2 | femoral_cartilage | cartilage |
| 3 | tibial_bone | bone |
| 4 | tibial_cartilage | cartilage |
| 5 | lateral_tibial_cartilage | cartilage |

Main idea: the default MedSAM 3D spacing of `1.5mm` makes thin cartilage nearly disappear. Training cartilage at `0.7mm` restores geometric detail, while bone remains at a coarser spacing to avoid cropping large structures with the fixed 128^3 input.

In [ ]:
# Cell 0 - Check GPU
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Tue Jun 23 05:54:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Install Dependencies

Run this cell once, then **restart the runtime/session immediately**. This is required because Torch, Numpy, and TorchIO are replaced in memory.

Notes:

- `torchio==0.18.92` keeps SAM-Med3D inference compatible with the original `CropOrPad` call.
- Training is patched later with a `SubjectsLoader = DataLoader` compatibility shim.

In [ ]:
# Cell 1 - Install dependencies, then restart runtime
# Do NOT install torchvision/torchaudio here. A version mismatch causes "torchvision::nms does not exist".
!pip uninstall -y torchvision torchaudio
!pip install -q --no-cache-dir "torchio==0.18.92" opencv-python-headless matplotlib prefetch_generator edt surface-distance medim SimpleITK nibabel pandas tqdm huggingface_hub monai

print("\n" + "="*72)
print("IMPORTANT: Runtime -> Restart session, then continue from Cell 2.")
print("Do not import torchvision or torchaudio for this workflow.")
print("="*72)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 164.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 484.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 270.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 726.9/726.9 MB 117.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 609.6/609.6 MB 105.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 183.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 276.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.4/260.4 MB 202.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.1/292.1 MB 170.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 88.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 MB 212.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 228.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip uninstall -y torchvision torchaudio
!pip install -q --no-cache-dir "torchio==0.18.92" opencv-python-headless matplotlib prefetch_generator edt surface-distance medim SimpleITK nibabel pandas tqdm huggingface_hub monai

Found existing installation: torchvision 0.22.0+cu128
Uninstalling torchvision-0.22.0+cu128:
  Successfully uninstalled torchvision-0.22.0+cu128
Found existing installation: torchaudio 2.7.0+cu128
Uninstalling torchaudio-2.7.0+cu128:
  Successfully uninstalled torchaudio-2.7.0+cu128


In [ ]:
import torch, torchio, monai
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("torchio:", torchio.__version__)
print("monai:", monai.__version__)

torch: 2.12.1+cu130
cuda: 13.0
cuda available: True
torchio: 0.18.92
monai: 1.6.0


## 2. Mount Drive, Configure Paths, Define Class Names

In [ ]:
# Cell 2 - Drive + global config
from google.colab import drive
drive.mount('/content/drive')

import os, os.path as osp
from glob import glob

# Main paths
RAW_ROOT = "/content/nnUNet_raw/Dataset001_KneeOA"
SAM_ROOT = "/content/SAM-Med3D"
DRIVE_ROOT = "/content/drive/MyDrive/SAM_Med3D"
WORK_DIR = f"{DRIVE_ROOT}/work_dir"
PRED_ROOT = f"{DRIVE_ROOT}/predictions"
BASE_CKPT = f"{DRIVE_ROOT}/ckpt/sam_med3d_turbo.pth"

# Hybrid task names/checkpoints
BONE_TASK = "oaizib_ft"                 # baseline/all-class 1.5mm checkpoint; can be replaced by bone-only task
CART_TASK = "oaizib_cart_07_e100"       # new 100-epoch cartilage model
BONE_CKPT = f"{WORK_DIR}/{BONE_TASK}/sam_model_dice_best.pth"
CART_CKPT = f"{WORK_DIR}/{CART_TASK}/sam_model_dice_best.pth"

# Inference/output folders
BONE_PRED_DIR = f"{PRED_ROOT}/oaizib_hybrid/bone_from_{BONE_TASK}_sp15"
CART_PRED_DIR = f"{PRED_ROOT}/oaizib_hybrid/cart_from_{CART_TASK}_sp07"
HYBRID_PRED_DIR = f"{PRED_ROOT}/oaizib_hybrid/final_hybrid"

CLASS_INFO = {
    1: {"key": "femoral_bone", "name": "femoral_bone", "group": "bone", "spacing": 1.5},
    2: {"key": "femoral_cartilage", "name": "femoral_cartilage", "group": "cartilage", "spacing": 0.7},
    3: {"key": "tibial_bone", "name": "tibial_bone", "group": "bone", "spacing": 1.5},
    4: {"key": "tibial_cartilage", "name": "tibial_cartilage", "group": "cartilage", "spacing": 0.7},
    5: {"key": "lateral_tibial_cartilage", "name": "lateral_tibial_cartilage", "group": "cartilage", "spacing": 0.7},
}
BONE_LABELS = [1, 3]
CART_LABELS = [2, 4, 5]
ALL_LABELS = [1, 2, 3, 4, 5]

for p in [DRIVE_ROOT, WORK_DIR, PRED_ROOT, BONE_PRED_DIR, CART_PRED_DIR, HYBRID_PRED_DIR, osp.dirname(BASE_CKPT)]:
    os.makedirs(p, exist_ok=True)

print("Class mapping:")
for k,v in CLASS_INFO.items():
    print(f"  {k}: {v['name']} | group={v['group']} | spacing={v['spacing']}mm")
print("\nBASE_CKPT:", BASE_CKPT)
print("BONE_CKPT:", BONE_CKPT)
print("CART_CKPT:", CART_CKPT)

Mounted at /content/drive
Class mapping:
  1: femoral_bone | group=bone | spacing=1.5mm
  2: femoral_cartilage | group=cartilage | spacing=0.7mm
  3: tibial_bone | group=bone | spacing=1.5mm
  4: tibial_cartilage | group=cartilage | spacing=0.7mm
  5: lateral_tibial_cartilage | group=cartilage | spacing=0.7mm

BASE_CKPT: /content/drive/MyDrive/SAM_Med3D/ckpt/sam_med3d_turbo.pth
BONE_CKPT: /content/drive/MyDrive/SAM_Med3D/work_dir/oaizib_ft/sam_model_dice_best.pth
CART_CKPT: /content/drive/MyDrive/SAM_Med3D/work_dir/oaizib_cart_07_e100/sam_model_dice_best.pth


## 3. Download Base Checkpoint and OAI-ZIB Data

If Hugging Face authentication is required in your environment, run `huggingface_hub.login()` manually. Do not store a personal token inside the notebook.

In [ ]:
# Cell 3 - Download SAM-Med3D checkpoint if missing
import urllib.request, os

if not osp.exists(BASE_CKPT):
    print("Downloading SAM-Med3D-turbo checkpoint (~360 MB)...")
    url = "https://huggingface.co/blueyo0/SAM-Med3D/resolve/main/sam_med3d_turbo.pth"
    urllib.request.urlretrieve(url, BASE_CKPT)
    print("Downloaded:", BASE_CKPT)
else:
    print("Checkpoint already exists:", BASE_CKPT)

Checkpoint already exists: /content/drive/MyDrive/SAM_Med3D/ckpt/sam_med3d_turbo.pth


In [ ]:
# Cell 4 - Download OAI-ZIB dataset if needed
from huggingface_hub import snapshot_download
import os, os.path as osp

os.makedirs(RAW_ROOT, exist_ok=True)
need_download = not (osp.exists(osp.join(RAW_ROOT, "imagesTr")) and osp.exists(osp.join(RAW_ROOT, "labelsTr")))
if need_download:
    snapshot_download(
        repo_id="YongchengYAO/OAIZIB-CM",
        repo_type="dataset",
        local_dir=RAW_ROOT,
        local_dir_use_symlinks=False,
    )
    print("Downloaded dataset to", RAW_ROOT)
else:
    print("Dataset folders already exist:", RAW_ROOT)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Downloaded dataset to /content/nnUNet_raw/Dataset001_KneeOA


In [ ]:
# Cell 5 - Unzip OAI-ZIB archives if images/labels folders are not present yet
import os, os.path as osp, zipfile

for name in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
    folder = osp.join(RAW_ROOT, name)
    zpath = osp.join(RAW_ROOT, f"{name}.zip")
    if not osp.exists(folder) and osp.exists(zpath):
        print("Unzipping", zpath)
        with zipfile.ZipFile(zpath, 'r') as zf:
            zf.extractall(RAW_ROOT)

for name in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
    folder = osp.join(RAW_ROOT, name)
    print(name, "exists=", osp.exists(folder), "n=", len(glob(osp.join(folder, "*.nii.gz"))) if osp.exists(folder) else 0)

Unzipping /content/nnUNet_raw/Dataset001_KneeOA/imagesTr.zip
Unzipping /content/nnUNet_raw/Dataset001_KneeOA/labelsTr.zip
Unzipping /content/nnUNet_raw/Dataset001_KneeOA/imagesTs.zip
Unzipping /content/nnUNet_raw/Dataset001_KneeOA/labelsTs.zip
imagesTr exists= True n= 404
labelsTr exists= True n= 404
imagesTs exists= True n= 103
labelsTs exists= True n= 103


## 4. Clone SAM-Med3D and Patch Compatibility Issues

The following patches handle the version issues observed during the experiments:

- TorchIO 0.18.x does not expose `tio.SubjectsLoader`, so we map it to PyTorch `DataLoader`. This is safe here because the training dataset returns plain tensor dictionaries.
- Newer MONAI DiceLoss expects the target tensor to be floating point, so `gt3D` is cast to `float`.

In [ ]:
# Cell 6 - Clone SAM-Med3D repo if needed
%cd /content
if not osp.exists(SAM_ROOT):
    !git clone https://github.com/uni-medical/SAM-Med3D.git
else:
    print("SAM-Med3D repo already exists:", SAM_ROOT)
%cd /content/SAM-Med3D

/content
Cloning into 'SAM-Med3D'...
remote: Enumerating objects: 604, done.
remote: Counting objects: 100% (239/239), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 604 (delta 188), reused 121 (delta 121), pack-reused 365 (from 3)
Receiving objects: 100% (604/604), 26.35 MiB | 28.38 MiB/s, done.
Resolving deltas: 100% (315/315), done.
/content/SAM-Med3D


In [ ]:
# Cell 7 - Patch train.py and data_loader.py for current Colab library versions
import os, re

# 1) Shim torchio SubjectsLoader -> DataLoader under torchio 0.18.x
p = f"{SAM_ROOT}/utils/data_loader.py"
s = open(p, encoding="utf-8").read()
anchor = "from torchio.data.io import sitk_to_nib\n"
shim = (
    "\n# Compatibility shim: torchio 0.18.x has no SubjectsLoader.\n"
    "# Dataset_Union_ALL returns dict tensors, so PyTorch DataLoader is sufficient.\n"
    "if not hasattr(tio, 'SubjectsLoader'):\n"
    "    tio.SubjectsLoader = DataLoader\n"
)
if "tio.SubjectsLoader = DataLoader" not in s:
    assert anchor in s, "Cannot find data_loader import anchor."
    s = s.replace(anchor, anchor + shim, 1)
    open(p, "w", encoding="utf-8").write(s)
    print("Patched data_loader.py: SubjectsLoader shim")
else:
    print("data_loader.py already patched")

# 2) MONAI DiceLoss target must be float
p = f"{SAM_ROOT}/train.py"
s = open(p, encoding="utf-8").read()
s2 = re.sub(r"self\.seg_loss\(\s*prev_masks\s*,\s*gt3D\s*\)", "self.seg_loss(prev_masks, gt3D.float())", s)
if s2 != s:
    open(p, "w", encoding="utf-8").write(s2)
    print("Patched train.py: gt3D.float() for seg_loss")
else:
    print("train.py already patched or pattern not found")

# Verify
s = open(f"{SAM_ROOT}/train.py", encoding="utf-8").read()
assert "self.seg_loss(prev_masks, gt3D.float())" in s, "train.py float patch missing"
print("Compatibility patches OK")

Patched data_loader.py: SubjectsLoader shim
Patched train.py: gt3D.float() for seg_loss
Compatibility patches OK


## 5. Prepare Per-Class Training Data

SAM-Med3D trains on binary class folders. This notebook prepares two possible datasets:

- `oaizib_bone_15`: labels 1 and 3 at 1.5mm spacing. This is optional if you want a dedicated bone model.
- `oaizib_cart_07`: labels 2, 4, and 5 at 0.7mm spacing. This is the main dataset for the 100-epoch cartilage model.

By default, the notebook trains only the cartilage model. Bone predictions are taken from the existing baseline checkpoint `oaizib_ft`.

In [ ]:
# Cell 8 - Prepare binary per-class data at requested spacing
import os, os.path as osp, shutil
from glob import glob
from tqdm import tqdm
import nibabel as nib
import torchio as tio

TRAIN_IMG_DIR = osp.join(RAW_ROOT, "imagesTr")
TRAIN_GT_DIR = osp.join(RAW_ROOT, "labelsTr")
assert osp.exists(TRAIN_IMG_DIR), TRAIN_IMG_DIR
assert osp.exists(TRAIN_GT_DIR), TRAIN_GT_DIR


def resample_nii(input_path, output_path, target_spacing=(1.5,1.5,1.5), n=None, reference_image=None, mode="linear"):
    subject = tio.Subject(img=tio.ScalarImage(input_path))
    resampled = tio.Resample(target=target_spacing, image_interpolation=mode)(subject)
    if n is not None:
        image = resampled.img
        td = image.data
        if isinstance(n, int):
            n = [n]
        for ni in n:
            td[td == ni] = -1
        td[td != -1] = 0
        td[td != 0] = 1
        save_image = tio.ScalarImage(tensor=td, affine=image.affine)
        save_image = tio.CropOrPad(reference_image.shape[1:])(save_image)
    else:
        save_image = resampled.img
    save_image.save(output_path)


def prepare_class_dataset(out_root, class_ids, spacing, overwrite=False):
    os.makedirs(out_root, exist_ok=True)
    img_cache = osp.join(out_root, "_img_resampled")
    os.makedirs(img_cache, exist_ok=True)
    cases = sorted(osp.basename(p).replace(".nii.gz", "") for p in glob(osp.join(TRAIN_GT_DIR, "*.nii.gz")))
    print(f"Preparing {len(cases)} cases at spacing={spacing}mm -> {out_root}")

    for idx in class_ids:
        cls_key = CLASS_INFO[idx]["key"]
        target_img_dir = osp.join(out_root, cls_key, "oaizib", "imagesTr")
        target_gt_dir = osp.join(out_root, cls_key, "oaizib", "labelsTr")
        os.makedirs(target_img_dir, exist_ok=True)
        os.makedirs(target_gt_dir, exist_ok=True)

        for case in tqdm(cases, desc=f"{idx}-{cls_key}"):
            img_in = osp.join(TRAIN_IMG_DIR, f"{case}_0000.nii.gz")
            gt_in = osp.join(TRAIN_GT_DIR, f"{case}.nii.gz")
            if not (osp.exists(img_in) and osp.exists(gt_in)):
                continue

            img_rs = osp.join(img_cache, f"{case}.nii.gz")
            if overwrite or not osp.exists(img_rs):
                resample_nii(img_in, img_rs, (spacing, spacing, spacing), mode="linear")

            # Skip tiny masks, same spirit as original prepare script (<10 mm^3)
            g = nib.load(gt_in)
            sp0 = tuple(g.header['pixdim'][1:4])
            volume_mm3 = (g.get_fdata() == idx).sum() * sp0[0] * sp0[1] * sp0[2]
            if volume_mm3 < 10:
                continue

            out_img = osp.join(target_img_dir, f"{case}.nii.gz")
            out_gt = osp.join(target_gt_dir, f"{case}.nii.gz")
            if overwrite or not osp.exists(out_gt):
                resample_nii(gt_in, out_gt, (spacing, spacing, spacing), n=idx,
                             reference_image=tio.ScalarImage(img_rs), mode="nearest")
            if overwrite or not osp.exists(out_img):
                shutil.copy(img_rs, out_img)

    valid_paths = []
    for idx in class_ids:
        cls_key = CLASS_INFO[idx]["key"]
        p = osp.join(out_root, cls_key, "oaizib")
        n_img = len(glob(osp.join(p, "imagesTr", "*.nii.gz")))
        n_gt = len(glob(osp.join(p, "labelsTr", "*.nii.gz")))
        valid_paths.append(p)
        print(f"  {idx} {CLASS_INFO[idx]['name']}: {n_img} images, {n_gt} labels")
    return valid_paths

BONE_DATA_ROOT = f"{SAM_ROOT}/data/oaizib_bone_15"
CART_DATA_ROOT = f"{SAM_ROOT}/data/oaizib_cart_07"

# Main required dataset: cartilage @0.7mm
cart_paths = prepare_class_dataset(CART_DATA_ROOT, CART_LABELS, spacing=0.7, overwrite=False)

# Optional dataset: bone @1.5mm, only needed if you want to train a bone-only model later
# bone_paths = prepare_class_dataset(BONE_DATA_ROOT, BONE_LABELS, spacing=1.5, overwrite=False)

Preparing 404 cases at spacing=0.7mm -> /content/SAM-Med3D/data/oaizib_cart_07



2-femoral_cartilage: 100%|██████████| 404/404 [04:21<00:00,  1.54it/s]

4-tibial_cartilage: 100%|██████████| 404/404 [01:26<00:00,  4.65it/s]

5-lateral_tibial_cartilage: 100%|██████████| 404/404 [01:27<00:00,  4.61it/s]

  2 femoral_cartilage: 404 images, 404 labels
  4 tibial_cartilage: 404 images, 404 labels
  5 lateral_tibial_cartilage: 404 images, 404 labels


In [ ]:
# Cell 9 - Point SAM-Med3D data_paths.py to cartilage folders only
from importlib import reload
import os.path as osp

cart_paths = [osp.join(CART_DATA_ROOT, CLASS_INFO[i]["key"], "oaizib") for i in CART_LABELS]
content = "img_datas = [\n" + "\n".join(f'    "{p}",' for p in cart_paths) + "\n]\n"
open(f"{SAM_ROOT}/utils/data_paths.py", "w", encoding="utf-8").write(content)

import sys, importlib
sys.path.insert(0, SAM_ROOT)
import utils.data_paths as dp
reload(dp)
print("img_datas:")
for p in dp.img_datas:
    print(" ", p)
assert len(dp.img_datas) == 3, dp.img_datas

img_datas:
  /content/SAM-Med3D/data/oaizib_cart_07/femoral_cartilage/oaizib
  /content/SAM-Med3D/data/oaizib_cart_07/tibial_cartilage/oaizib
  /content/SAM-Med3D/data/oaizib_cart_07/lateral_tibial_cartilage/oaizib


In [ ]:
# Cell 10 - Sanity-check training dataset before spending hours training
%cd /content/SAM-Med3D
import torchio as tio
from importlib import reload
import utils.data_paths as dp; reload(dp)
from utils.data_loader import Dataset_Union_ALL

ds = Dataset_Union_ALL(paths=dp.img_datas, transform=tio.Compose([
    tio.ToCanonical(),
    tio.CropOrPad(mask_name='label', target_shape=(128,128,128)),
]), threshold=0)
print("num samples:", len(ds))
assert len(ds) > 0, "0 samples: check data_paths.py and data folder structure"
s = ds[0]
print("image shape:", tuple(s["image"].shape), "label shape:", tuple(s["label"].shape), "label sum:", float(s["label"].sum()))
assert tuple(s["image"].shape) == (1,128,128,128)

/content/SAM-Med3D
num samples: 1212
image shape: (1, 128, 128, 128) label shape: (1, 128, 128, 128) label sum: 40905.0


## 6. Train Cartilage Model @0.7mm for 100 Epochs

This cell fine-tunes the cartilage model from `sam_med3d_turbo.pth` for **100 epochs**.

- Task: `oaizib_cart_07_e100`
- Data: labels 2, 4, and 5 only
- Spacing: 0.7mm offline-preprocessed data
- Input: fixed 128^3 volume, matching the SAM-Med3D positional embedding

If training runs out of memory, use `--batch_size 2 --accumulation_steps 20`.

In [ ]:
# Patch SAM-Med3D package init to avoid unused torchvision dependency
p = "/content/SAM-Med3D/segment_anything/__init__.py"

backup = open(p, encoding="utf-8").read()
open(p + ".bak", "w", encoding="utf-8").write(backup)

open(p, "w", encoding="utf-8").write("""
# Minimal init for SAM-Med3D training/inference.
# Avoid importing automatic_mask_generator because it requires torchvision.ops,
# which is not needed for this 3D fine-tuning pipeline.

__all__ = []
""")

print("Patched:", p)

Patched: /content/SAM-Med3D/segment_anything/__init__.py


In [ ]:
# Cell 11 - Train cartilage model, 100 epochs
%cd /content/SAM-Med3D
assert osp.exists(BASE_CKPT), BASE_CKPT
!python train.py \
    --task_name "oaizib_cart_07_e100" \
    --checkpoint /content/drive/MyDrive/SAM_Med3D/ckpt/sam_med3d_turbo.pth \
    --work_dir /content/drive/MyDrive/SAM_Med3D/work_dir \
    --batch_size 4 \
    --accumulation_steps 10 \
    --num_workers 2 \
    --num_epochs 100 \
    --step_size 50 \
    --gamma 0.1 \
    --lr 8e-5 \
    --img_size 128 \
    --gpu_ids 0 \
    --allow_partial_weight \
    --click_type random

/content/SAM-Med3D
Loaded checkpoint from /content/drive/MyDrive/SAM_Med3D/ckpt/sam_med3d_turbo.pth (epoch 0)
Epoch: 0/99
  3% 10/303 [00:05<01:48,  2.69it/s]Epoch: 0, Step: 10, Loss: 8.377378892898559, Dice: 0.39506861567497253
  7% 20/303 [00:08<01:38,  2.86it/s]Epoch: 0, Step: 20, Loss: 6.455065584182739, Dice: 0.47888368368148804
 10% 30/303 [00:12<01:34,  2.89it/s]Epoch: 0, Step: 30, Loss: 7.005850601196289, Dice: 0.3387588560581207
 13% 40/303 [00:15<01:28,  2.98it/s]Epoch: 0, Step: 40, Loss: 6.8392143726348875, Dice: 0.2361176759004593
 17% 50/303 [00:19<01:27,  2.90it/s]Epoch: 0, Step: 50, Loss: 6.035984325408935, Dice: 0.42850908637046814
 20% 60/303 [00:22<01:22,  2.96it/s]Epoch: 0, Step: 60, Loss: 5.158822059631348, Dice: 0.3917829394340515
 23% 70/303 [00:25<01:17,  3.00it/s]Epoch: 0, Step: 70, Loss: 4.933516025543213, Dice: 0.4354880452156067
 26% 80/303 [00:29<01:14,  2.99it/s]Epoch: 0, Step: 80, Loss: 4.763349294662476, Dice: 0.4328927993774414
 30% 90/303 [00:32<01:10, 

## 7. Hybrid Inference

The hybrid pipeline runs two inference passes:

1. `bone model @1.5mm` -> use labels 1 and 3.
2. `cartilage model @0.7mm` -> use labels 2, 4, and 5.

The final mask is merged by assigning bone first and cartilage second. Cartilage therefore overrides bone in boundary-contact regions.

In [ ]:
# Cell 12 - Run bone inference @1.5mm and cartilage inference @0.7mm
%cd /content/SAM-Med3D
import os, os.path as osp
from glob import glob
from tqdm import tqdm
import medim
from utils.infer_utils import validate_paired_img_gt

NUM_CLICKS = 10
IMG_TS_DIR = osp.join(RAW_ROOT, "imagesTs")
GT_TS_DIR = osp.join(RAW_ROOT, "labelsTs")
assert osp.exists(IMG_TS_DIR), IMG_TS_DIR
assert osp.exists(GT_TS_DIR), GT_TS_DIR
assert osp.exists(BONE_CKPT), f"Missing bone checkpoint: {BONE_CKPT}. Train/use baseline model first."
assert osp.exists(CART_CKPT), f"Missing cartilage checkpoint: {CART_CKPT}. Run Cell 11 first."

os.makedirs(BONE_PRED_DIR, exist_ok=True)
os.makedirs(CART_PRED_DIR, exist_ok=True)

gt_list = sorted(glob(osp.join(GT_TS_DIR, "*.nii.gz")))
print("Test cases:", len(gt_list))

print("\nLoading bone model:", BONE_CKPT)
bone_model = medim.create_model("SAM-Med3D", pretrained=True, checkpoint_path=BONE_CKPT)
for gt_f in tqdm(gt_list, desc="Bone inference @1.5mm"):
    case = osp.basename(gt_f).replace(".nii.gz", "")
    img_f = osp.join(IMG_TS_DIR, f"{case}_0000.nii.gz")
    out_f = osp.join(BONE_PRED_DIR, f"{case}.nii.gz")
    if osp.exists(out_f):
        continue
    validate_paired_img_gt(bone_model, img_f, gt_f, out_f,
                           num_clicks=NUM_CLICKS, target_spacing=(1.5,1.5,1.5), crop_size=128)

del bone_model
import torch; torch.cuda.empty_cache()

print("\nLoading cartilage model:", CART_CKPT)
cart_model = medim.create_model("SAM-Med3D", pretrained=True, checkpoint_path=CART_CKPT)
for gt_f in tqdm(gt_list, desc="Cartilage inference @0.7mm"):
    case = osp.basename(gt_f).replace(".nii.gz", "")
    img_f = osp.join(IMG_TS_DIR, f"{case}_0000.nii.gz")
    out_f = osp.join(CART_PRED_DIR, f"{case}.nii.gz")
    if osp.exists(out_f):
        continue
    validate_paired_img_gt(cart_model, img_f, gt_f, out_f,
                           num_clicks=NUM_CLICKS, target_spacing=(0.7,0.7,0.7), crop_size=128)

del cart_model
torch.cuda.empty_cache()
print("Done inference.")

/content/SAM-Med3D
Test cases: 103

Loading bone model: /content/drive/MyDrive/SAM_Med3D/work_dir/oaizib_ft/sam_model_dice_best.pth
creating model SAM-Med3D
try to load pretrained weights from /content/drive/MyDrive/SAM_Med3D/work_dir/oaizib_ft/sam_model_dice_best.pth



Bone inference @1.5mm: 100%|██████████| 103/103 [05:51<00:00,  3.41s/it]



Loading cartilage model: /content/drive/MyDrive/SAM_Med3D/work_dir/oaizib_cart_07_e100/sam_model_dice_best.pth
creating model SAM-Med3D
try to load pretrained weights from /content/drive/MyDrive/SAM_Med3D/work_dir/oaizib_cart_07_e100/sam_model_dice_best.pth



Cartilage inference @0.7mm: 100%|██████████| 103/103 [05:03<00:00,  2.94s/it]

Done inference.


In [ ]:
# Cell 13 - Merge bone + cartilage predictions into hybrid multiclass masks
import os, os.path as osp
from glob import glob
from tqdm import tqdm
import numpy as np
import nibabel as nib

os.makedirs(HYBRID_PRED_DIR, exist_ok=True)
gt_list = sorted(glob(osp.join(GT_TS_DIR, "*.nii.gz")))

for gt_f in tqdm(gt_list, desc="Merging hybrid masks"):
    case = osp.basename(gt_f).replace(".nii.gz", "")
    out_f = osp.join(HYBRID_PRED_DIR, f"{case}.nii.gz")
    if osp.exists(out_f):
        continue

    bone_f = osp.join(BONE_PRED_DIR, f"{case}.nii.gz")
    cart_f = osp.join(CART_PRED_DIR, f"{case}.nii.gz")
    assert osp.exists(bone_f), bone_f
    assert osp.exists(cart_f), cart_f

    gt_img = nib.load(gt_f)
    bone = np.asarray(nib.load(bone_f).dataobj).astype(np.int16)
    cart = np.asarray(nib.load(cart_f).dataobj).astype(np.int16)
    hybrid = np.zeros_like(bone, dtype=np.int16)

    # Bone first
    for label in BONE_LABELS:
        hybrid[bone == label] = label

    # Cartilage overrides bone at boundaries
    for label in CART_LABELS:
        hybrid[cart == label] = label

    nib.save(nib.Nifti1Image(hybrid, gt_img.affine, gt_img.header), out_f)

print("Hybrid predictions:", HYBRID_PRED_DIR)


Merging hybrid masks: 100%|██████████| 103/103 [00:54<00:00,  1.87it/s]

Hybrid predictions: /content/drive/MyDrive/SAM_Med3D/predictions/oaizib_hybrid/final_hybrid


## 8. Named Metrics and Comparison Tables

In [ ]:
# Cell 14 - Compute metrics and print class names clearly
%cd /content/SAM-Med3D
import os.path as osp
from glob import glob
from tqdm import tqdm
from utils.metric_utils import compute_metrics


def normalize_metric_name(k):
    k = str(k).lower()
    if k in ["dice", "dsc"]: return "DSC"
    if k == "nsd": return "NSD"
    return k.upper()


def metric_value(vals, names):
    for n in names:
        if n in vals: return vals[n]
        if n.lower() in vals: return vals[n.lower()]
        if n.upper() in vals: return vals[n.upper()]
    return None


def compute_folder_metrics(pred_dir, title):
    gt_list = sorted(glob(osp.join(GT_TS_DIR, "*.nii.gz")))
    all_results = {}
    for gt_f in tqdm(gt_list, desc=f"Metrics: {title}"):
        case = osp.basename(gt_f).replace(".nii.gz", "")
        pred_f = osp.join(pred_dir, f"{case}.nii.gz")
        if not osp.exists(pred_f):
            print("Skip missing pred:", case)
            continue
        all_results[case] = compute_metrics(gt_path=gt_f, pred_path=pred_f, metrics=['dsc','nsd'], classes=None)
    return all_results


def summarize_named(all_results, title):
    import numpy as np
    per_class = {c: {"DSC": [], "NSD": []} for c in ALL_LABELS}
    for case, case_res in all_results.items():
        for cls, vals in case_res.items():
            c = int(cls)
            if c not in per_class: continue
            dsc = metric_value(vals, ["dsc", "dice", "DSC", "Dice"])
            nsd = metric_value(vals, ["nsd", "NSD"])
            if dsc is not None: per_class[c]["DSC"].append(float(dsc))
            if nsd is not None: per_class[c]["NSD"].append(float(nsd))

    print("\n" + "="*92)
    print(title)
    print("="*92)
    print(f"{'Label':<6} {'Class':<46} {'Group':<10} {'DSC':>8} {'NSD':>8}")
    print("-"*92)
    rows = {}
    for c in ALL_LABELS:
        dsc = float(np.mean(per_class[c]["DSC"])) if per_class[c]["DSC"] else float('nan')
        nsd = float(np.mean(per_class[c]["NSD"])) if per_class[c]["NSD"] else float('nan')
        rows[c] = {"DSC": dsc, "NSD": nsd}
        print(f"{c:<6} {CLASS_INFO[c]['name']:<46} {CLASS_INFO[c]['group']:<10} {dsc:8.4f} {nsd:8.4f}")
    avg_dsc = np.nanmean([rows[c]["DSC"] for c in ALL_LABELS])
    avg_nsd = np.nanmean([rows[c]["NSD"] for c in ALL_LABELS])
    cart_dsc = np.nanmean([rows[c]["DSC"] for c in CART_LABELS])
    cart_nsd = np.nanmean([rows[c]["NSD"] for c in CART_LABELS])
    print("-"*92)
    print(f"Average all classes: DSC={avg_dsc:.4f}, NSD={avg_nsd:.4f}")
    print(f"Average cartilage only: DSC={cart_dsc:.4f}, NSD={cart_nsd:.4f}")
    return rows

hybrid_results = compute_folder_metrics(HYBRID_PRED_DIR, "Hybrid")
hybrid_rows = summarize_named(hybrid_results, f"SAM-Med3D Hybrid: bone@1.5mm + cartilage@0.7mm ({len(hybrid_results)} cases)")

/content/SAM-Med3D



Metrics: Hybrid: 100%|██████████| 103/103 [04:07<00:00,  2.40s/it]


SAM-Med3D Hybrid: bone@1.5mm + cartilage@0.7mm (103 cases)
Label  Class                                          Group           DSC      NSD
--------------------------------------------------------------------------------------------
1      femoral_bone                                   bone         0.8127   0.3059
2      femoral_cartilage                              cartilage    0.8255   0.9440
3      tibial_bone                                    bone         0.8327   0.3086
4      tibial_cartilage                               cartilage    0.7532   0.9358
5      lateral_tibial_cartilage                       cartilage    0.7859   0.9377
--------------------------------------------------------------------------------------------
Average all classes: DSC=0.8020, NSD=0.6864
Average cartilage only: DSC=0.7882, NSD=0.9392


In [ ]:
# Cell 15 - Compare against known baselines with class names
# Fill/update these if you rerun another baseline.
BASELINE_MEDSAM_15 = {
    1: {"DSC": 0.9494, "NSD": 0.7318},
    2: {"DSC": 0.6714, "NSD": 0.7554},
    3: {"DSC": 0.9481, "NSD": 0.7498},
    4: {"DSC": 0.3816, "NSD": 0.4118},
    5: {"DSC": 0.2403, "NSD": 0.3646},
}
NNUNET_V2 = {
    1: {"DSC": 0.9580},
    2: {"DSC": 0.8820},
    3: {"DSC": 0.9840},
    4: {"DSC": 0.8340},
    5: {"DSC": 0.8670},
}

print("\n" + "="*118)
print("Comparison: baseline MedSAM 1.5mm vs hybrid vs nnUNet v2")
print("="*118)
print(f"{'Label':<6} {'Class':<46} {'MedSAM 1.5 DSC':>15} {'Hybrid DSC':>12} {'Delta Hybrid':>10} {'nnUNet DSC':>12} {'Hybrid NSD':>12}")
print("-"*118)
for c in ALL_LABELS:
    base = BASELINE_MEDSAM_15[c]["DSC"]
    hyb = hybrid_rows[c]["DSC"]
    nn = NNUNET_V2[c]["DSC"]
    hnsd = hybrid_rows[c]["NSD"]
    print(f"{c:<6} {CLASS_INFO[c]['name']:<46} {base:15.4f} {hyb:12.4f} {hyb-base:10.4f} {nn:12.4f} {hnsd:12.4f}")

import numpy as np
base_cart = np.mean([BASELINE_MEDSAM_15[c]["DSC"] for c in CART_LABELS])
hyb_cart = np.mean([hybrid_rows[c]["DSC"] for c in CART_LABELS])
print("-"*118)
print(f"Cartilage mean DSC: MedSAM 1.5mm={base_cart:.4f} | Hybrid={hyb_cart:.4f} | Delta={hyb_cart-base_cart:.4f}")


Comparison: baseline MedSAM 1.5mm vs hybrid vs nnUNet v2
Label  Class                                           MedSAM 1.5 DSC   Hybrid DSC Delta Hybrid   nnUNet DSC   Hybrid NSD
----------------------------------------------------------------------------------------------------------------------
1      femoral_bone                                            0.9494       0.8127    -0.1367       0.9580       0.3059
2      femoral_cartilage                                       0.6714       0.8255     0.1541       0.8820       0.9440
3      tibial_bone                                             0.9481       0.8327    -0.1154       0.9840       0.3086
4      tibial_cartilage                                        0.3816       0.7532     0.3716       0.8340       0.9358
5      lateral_tibial_cartilage                                0.2403       0.7859     0.5456       0.8670       0.9377
-----------------------------------------------------------------------------------------------------

## 9. Optional: Train Bone-Only Model @1.5mm

By default, this notebook uses the baseline/all-class checkpoint `oaizib_ft` for bone classes. If you want a cleaner hybrid setup, you can train a dedicated bone-only model for labels 1 and 3 at 1.5mm using the optional cell below.

In [ ]:
# Optional Cell - Prepare and train bone-only model @1.5mm
# Uncomment if you want a dedicated bone model instead of using BONE_TASK=oaizib_ft.

# bone_paths = prepare_class_dataset(BONE_DATA_ROOT, BONE_LABELS, spacing=1.5, overwrite=False)
# content = "img_datas = [\n" + "\n".join(f'    "{p}",' for p in bone_paths) + "\n]\n"
# open(f"{SAM_ROOT}/utils/data_paths.py", "w", encoding="utf-8").write(content)
# print(content)

# %cd /content/SAM-Med3D
# !python train.py \
#     --task_name "oaizib_bone_15_e100" \
#     --checkpoint /content/drive/MyDrive/SAM_Med3D/ckpt/sam_med3d_turbo.pth \
#     --work_dir /content/drive/MyDrive/SAM_Med3D/work_dir \
#     --batch_size 4 \
#     --accumulation_steps 10 \
#     --num_workers 2 \
#     --num_epochs 100 \
#     --step_size 50 \
#     --gamma 0.1 \
#     --lr 8e-5 \
#     --img_size 128 \
#     --gpu_ids 0 \
#     --allow_partial_weight \
#     --click_type random